In [ ]:
# Jupyter Notebook -- Model Training Script: Preparing Python Scripts For The Model Training Process
import numpy as np
import os
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.initializers import HeNormal
from tensorflow.keras.regularizers import l1_l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import LabelEncoder
from preprocessing import target
from plot_results import plot_confusion_matrix, plot_classification_report

* With `Sequential`, we will build a neural network model by stacking layers in a sequential manner. It provides a simple, linear way to define models layer by layer.
  
* With `Dense`, we will create fully connected layers for our neural network. It’s the most common layer in deep learning, where every neuron in one layer is connected to every neuron in the next layer.
  
* With `Dropout`, we can apply regularization by randomly dropping a fraction of neurons during training, which helps prevent overfitting.

* With `Input`, we will define the input layer and the shape of the input data. It explicitly sets the input shape of the model, especially for the first layer.

* With `EarlyStopping`, we can stop the training process early if the model stops improving, which helps avoid overfitting and saves computation time.

* With `HeNormal`, we will initialize the weights of our layers using He Normal initialization, which is ideal for layers using ReLU activations to ensure proper weight scaling.

* With `l1_l2`, we will apply L1 and L2 regularization to our layers, which helps prevent overfitting by penalizing large weights in the model.

* With `Adam`, we will use the Adam optimizer to adjust the learning rate during training, ensuring more efficient and faster convergence.

* With `to_categorical`, we can convert integer labels into one-hot encoded vectors, which is necessary for multi-class classification tasks.

* With `LabelEncoder`, we can convert string labels into numeric form, which is essential for machine learning models that only work with numeric data.

* With `plot_confusion_matrix` and `plot_classification_report`, I will visualize the outputs using the model output data using these functions that I will create in my file named plot_results.py, which I will write tomorrow.

In [2]:
# Creating Model Func
def create_model(input_shape, num_classes, dropout_rate=0.2, l1_reg=0.01, l2_reg=0.01):
    initializer = HeNormal()
    model = Sequential([
        Input(shape=(input_shape,)),
        Dense(256, activation='relu', kernel_regularizer=l1_l2(l1=l1_reg, l2=l2_reg), kernel_initializer=initializer),
        Dropout(dropout_rate),
        Dense(128, activation='relu', kernel_regularizer=l1_l2(l1=l1_reg, l2=l2_reg), kernel_initializer=initializer),
        Dropout(dropout_rate),
        Dense(num_classes, activation='softmax', kernel_initializer=initializer)
    ])

    optimizer = Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    return model

* `Input(shape=(input_shape,))` specifies the size of the input data. The size of the input data is equal to the number of features.
* Since speed and simplicity are important, I used `ReLU` in the hidden layers.
* Since our `Category` value in our data set contains more than two values, we can say that it is a multiclass problem and in this case we need to use `softmax` in the output layer.
* `learning_rate` is used as default, I will evaluate later.
* `l1 and l2` are used as 0.01, I will evaluate later.
* I used `categorical_crossentropy` as a loss parameter for getting prediction values of classes.
* I used `accuracy` as a metrics because we will evaluate the model with its accuracy.
* I used `dropout_rate = 0.2` this means that 20% of the data used cannot be used in the next data, reducing the possibility of overfitting. I will evaluate later.

In [ ]:
# Defining number of classes
num_classes = len(target)

In [ ]:
# Preparing for Label Encoding
label_encoder = LabelEncoder()
label_encoder.fit(target)

Loading .npy files which are prepared at preprocessing.py file : 

In [ ]:
# Loading data
current_directory = os.getcwd()
X_train = np.load(os.path.join(current_directory, 'X_train_scaled.npy'))
y_train = np.load(os.path.join(current_directory, 'y_train.npy'), allow_pickle=True)
X_val = np.load(os.path.join(current_directory, 'X_val_scaled.npy'))
y_val = np.load(os.path.join(current_directory, 'y_val.npy'), allow_pickle=True)
X_test = np.load(os.path.join(current_directory, 'X_test_scaled.npy'))
y_test = np.load(os.path.join(current_directory, 'y_test.npy'), allow_pickle=True)

In [ ]:
# Converting string labels to numeric labels
y_train_encoded = label_encoder.transform(y_train)
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

In [ ]:
# Encode targets with one-hot encoding
y_train_categorical = to_categorical(y_train_encoded, num_classes=num_classes)
y_val_categorical = to_categorical(y_val_encoded, num_classes=num_classes)
y_test_categorical = to_categorical(y_test_encoded, num_classes=num_classes)

In [ ]:
# Creating model
model = create_model(X_train.shape[1], num_classes=num_classes)

In [ ]:
# Setting EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [ ]:
# Train the model
history = model.fit(X_train, y_train_categorical, epochs=30, validation_data=(X_val, y_val_categorical), callbacks=[early_stopping])

In [ ]:
# Save the model
model.save('website_category_model.keras')

In [ ]:
# Evaluating performance at Test's dataset
y_test_pred = model.predict(X_test)
y_test_pred_classes = np.argmax(y_test_pred, axis=1)
y_test_true_classes = np.argmax(y_test_categorical, axis=1)

In [ ]:
# Creating confusion_matrix and classification_report
plot_confusion_matrix(y_test_true_classes, y_test_pred_classes, label_encoder)
plot_classification_report(y_test_true_classes, y_test_pred_classes, label_encoder)